# AFTERIMAGE · Mensuel tous États/provinces : contrôles + livrables
N'utilise pas Earth Engine (seulement les CSV du Drive). Exécuter la cellule ci-dessous.

In [1]:
# === Mensuel tous États/provinces : dépliage + contrôles + livrables (ne nécessite PAS Earth Engine) ===
import glob, os, json, hashlib
import numpy as np, pandas as pd
from google.colab import drive
drive.mount('/content/drive')

BANDS = ['D_aqua', 'O_aqua', 'D_viirs', 'O_viirs', 'O_common',
         'D_aqua_common', 'D_viirs_common', 'D_aqua_nh', 'D_viirs_nh']
SRC = '/content/drive/MyDrive/AFTERIMAGE_EE_v2'
OUT = '/content/drive/MyDrive/AFTERIMAGE/ee/checked_v2'
os.makedirs(OUT, exist_ok=True)

# 1. Déplier les fichiers (la colonne "sums" contient les 9 comptages séparés par ;)
files = sorted(glob.glob(f'{SRC}/afterimage_all_monthly_*.csv'))
assert not any(f.endswith(('_2020.csv', '_2021.csv')) for f in files), 'ERREUR : 2020-2021 ne doivent pas être ici'
parts = []
for p in files:
    df = pd.read_csv(p)
    vals = df['sums'].astype(str).str.split(';', expand=True).astype(float)
    vals.columns = BANDS
    parts.append(pd.concat([df[['ADM1_CODE', 'month', 'first_day', 'n_days', 'aqua_days', 'viirs_days']], vals], axis=1))
m = pd.concat(parts, ignore_index=True)
m['ADM1_CODE'] = m['ADM1_CODE'].astype(int)

# 2. Noms des États/provinces (depuis le GeoJSON GAUL enregistré par la cellule FIRMS, sinon Earth Engine)
geo = '/content/drive/MyDrive/AFTERIMAGE/firms/blocks/blocks_gaul2015_simplified1km.geojson'
if os.path.exists(geo):
    import geopandas as gpd
    names = pd.DataFrame(gpd.read_file(geo).drop(columns='geometry'))[['ADM0_NAME', 'ADM1_CODE', 'ADM1_NAME']]
else:
    import ee
    ee.Initialize(project='inlaid-reactor-433400-d6')
    fc = (ee.FeatureCollection('FAO/GAUL/2015/level1')
          .filter(ee.Filter.inList('ADM0_NAME', ['United States of America', 'Canada']))
          .select(['ADM0_NAME', 'ADM1_CODE', 'ADM1_NAME']).map(lambda f: f.setGeometry(None)))
    names = pd.DataFrame([f['properties'] for f in fc.getInfo()['features']])
names['ADM1_CODE'] = names['ADM1_CODE'].astype(int)
m = m.merge(names.drop_duplicates('ADM1_CODE'), on='ADM1_CODE', how='left')

# 3. Contrôles AVANT de marquer les valeurs indisponibles
issues = []
for d, o in [('D_aqua', 'O_aqua'), ('D_viirs', 'O_viirs'), ('D_aqua_common', 'O_common'),
             ('D_viirs_common', 'O_common'), ('D_aqua_nh', 'D_aqua'), ('D_viirs_nh', 'D_viirs')]:
    bad = int((m[d] > m[o]).sum())
    if bad: issues.append(f'{bad} lignes avec {d} > {o}')
bad = int((m['O_common'] > m[['O_aqua', 'O_viirs']].min(axis=1)).sum())
if bad: issues.append(f'{bad} lignes avec O_common > min(O_aqua, O_viirs)')
dup = int(m.duplicated(['ADM1_CODE', 'month']).sum())
if dup: issues.append(f'{dup} doublons bloc-mois')
per_block = m.groupby('ADM1_CODE')['month'].nunique()
if per_block.nunique() > 1: issues.append(f'nombre de mois différent selon les blocs : {per_block.value_counts().to_dict()}')
if m['ADM1_NAME'].isna().any(): issues.append(f"{m['ADM1_NAME'].isna().sum()} lignes sans nom de bloc")

# 4. Indisponible = vide (jamais zéro) ; mois partiel signalé
m.loc[m['aqua_days'] == 0, ['D_aqua', 'O_aqua', 'D_aqua_nh']] = np.nan
m.loc[m['viirs_days'] == 0, ['D_viirs', 'O_viirs', 'D_viirs_nh']] = np.nan
m.loc[(m['aqua_days'] == 0) | (m['viirs_days'] == 0), ['O_common', 'D_aqua_common', 'D_viirs_common']] = np.nan
m['days_in_month'] = pd.to_datetime(m['month']).dt.days_in_month
m['aqua_complete'] = m['aqua_days'] == m['days_in_month']
m['viirs_complete'] = m['viirs_days'] == m['days_in_month']
for s in ['aqua', 'viirs']:
    m[f'R_{s}'] = np.where(m[f'O_{s}'] > 0, 1000 * m[f'D_{s}'] / m[f'O_{s}'], np.nan)   # O = 0 -> indisponible

cols = ['ADM0_NAME', 'ADM1_CODE', 'ADM1_NAME', 'month', 'days_in_month', 'aqua_days', 'viirs_days',
        'aqua_complete', 'viirs_complete'] + BANDS + ['R_aqua', 'R_viirs']
m = m[cols].sort_values(['ADM1_CODE', 'month'])

# 5. Livrables
m.to_parquet(f'{OUT}/all_blocks_monthly.parquet', index=False)          # P3 : carte + calendrier raw_only
rim = m[m['ADM1_NAME'].isin(['California', 'British Columbia']) & m['month'].between('2016-01', '2019-12')]
rim.to_csv(f'{OUT}/RIM_dev_monthly_CA_BC_2016_2019.csv', index=False)  # P1 : années de développement seulement

def sha(p): return hashlib.sha256(open(p, 'rb').read()).hexdigest()
report = {'source_files': [{'file': os.path.basename(p), 'sha256': sha(p)} for p in files],
          'rows': len(m), 'blocks': int(m['ADM1_CODE'].nunique()),
          'months': [m['month'].min(), m['month'].max()], 'rim_rows': len(rim), 'issues': issues,
          'rules': 'land QA bits0-1==2; fire FireMask 7-9; O = FireMask 5 or fire; common = usable in both same day; '
                   'empty month for a sensor = null; R = 1000*D/O with O=0 -> null. 2020-2021 sealed (P5).'}
json.dump(report, open(f'{OUT}/check_report_monthly.json', 'w'), indent=2)

print('Lignes :', len(m), '| blocs :', m['ADM1_CODE'].nunique(), '| mois :', m['month'].min(), '->', m['month'].max())
print('Contrôles :', 'OK' if not issues else issues)
print('Rim :', len(rim), 'lignes ->', f'{OUT}/RIM_dev_monthly_CA_BC_2016_2019.csv')


Mounted at /content/drive
Lignes : 16512 | blocs : 64 | mois : 2002-07 -> 2025-12
Contrôles : OK
Rim : 48 lignes -> /content/drive/MyDrive/AFTERIMAGE/ee/checked_v2/RIM_dev_monthly_CA_BC_2016_2019.csv


In [2]:
print(rim['ADM1_NAME'].value_counts())
print(m[m['ADM1_NAME'].str.contains('Brit|Calif', case=False, na=False)]['ADM1_NAME'].unique())

ADM1_NAME
California    48
Name: count, dtype: int64
['British Columbia / Colombie-Britannique' 'California']


In [3]:
FOCUS = ['California', 'British Columbia / Colombie-Britannique']
rim = m[m['ADM1_NAME'].isin(FOCUS) & m['month'].between('2016-01', '2019-12')]
rim.to_csv(f'{OUT}/RIM_dev_monthly_CA_BC_2016_2019.csv', index=False)
print(rim['ADM1_NAME'].value_counts())

ADM1_NAME
British Columbia / Colombie-Britannique    48
California                                 48
Name: count, dtype: int64


In [4]:
r = json.load(open(f'{OUT}/check_report_monthly.json'))
r['rim_rows'] = len(rim); r['focus_blocks'] = FOCUS
json.dump(r, open(f'{OUT}/check_report_monthly.json', 'w'), indent=2)
print('Rapport mis à jour :', r['rim_rows'], 'lignes')

Rapport mis à jour : 96 lignes


In [5]:
dev_all = m[m['month'].between('2016-01', '2019-12')]
dev_all.to_csv(f'{OUT}/RIM_dev_monthly_ALL_2016_2019.csv', index=False)
print(len(dev_all), 'lignes |', dev_all['ADM1_CODE'].nunique(), 'régions')

3072 lignes | 64 régions
